# Orquestrador da Camada Silver (Silver Layer Orchestrator)

Executa todas as cargas da camada Silver em paralelo usando threads no cluster do Databricks.

In [ ]:
%run ../0_Config/0-Init

In [ ]:
def run_notebook(notebook_name: str) -> None:
    dbutils.notebook.run(notebook_name, 0)

In [ ]:
job_list = [
    "001-atendimento_ocorrencias",
    "002-cadastro_produtos",
    "003-comercial_canais",
    "004-crm_clientes",
    "005-erp_pedidos_cabecalho",
    "006-erp_pedidos_itens",
    "007-legado_regioes",
    "008-logistica_entregas",
    "009-vendedores"
]

In [ ]:
from threading import Thread
from queue import Queue
import time

q = Queue()
for job in job_list:
    q.put(job)

def worker():
    while not q.empty():
        job_name = q.get()
        print(f"▶ Iniciando: {job_name}")
        start = time.time()
        try:
            run_notebook(job_name)
            print(f"✓ Concluído: {job_name} em {time.time() - start:.1f}s")
        except Exception as e:
            print(f"✗ Falha: {job_name} | Erro: {e}")
        finally:
            q.task_done()

for _ in range(5):
    t = Thread(target=worker)
    t.daemon = True
    t.start()

q.join()
print("Todas as cargas da Silver foram executadas.")